# Decoding Teacher Sentiment

A multi-task 1D CNN that predicts Quality & Difficulty scores from RateMyProfessor reviews, with LIME explainability.

---

## Cell 1: Scrape RateMyProfessors.com (optional)
Run this cell to scrape fresh data from RateMyProfessors.com via their GraphQL API.
If you already have the CSV, skip to Cell 2.

> Based on the code from [ratemyprofessor-api](https://github.com/ppannuta/ratemyprofessor-api) by ppannuta.

In [ ]:
# !pip install curl_cffi pandas

# Run this in a separate cell at the top if you haven't yet:
# !pip install curl_cffi pandas

from curl_cffi import requests
import pandas as pd
import base64
import time

# --- PROFESSOR CLASS ---
class Professor:
    def __init__(self, ratemyprof_id: str, first_name: str, last_name: str, num_of_ratings: int, overall_rating):
        self.ratemyprof_id = ratemyprof_id
        self.name = f"{first_name} {last_name}"
        self.num_of_ratings = num_of_ratings
        self.overall_rating = 0.0 if self.num_of_ratings < 1 else float(overall_rating)

# --- MAIN API CLASS ---
class RateMyProfApi:
    def __init__(self, school_id: str):
        self.UniversityId = school_id
        school_string = f"School-{self.UniversityId}"
        self.graphql_school_id = base64.b64encode(school_string.encode('utf-8')).decode('utf-8')

        self.headers = {
            "Authorization": "Basic dGVzdDp0ZXN0",
            "Content-Type": "application/json"
        }

        self.professors = self.scrape_professors()
        self.professorlist = list(self.professors.values())

    def scrape_professors(self):
        professors = dict()

        # Added resultCount to track the total pool
        query = """
        query TeacherSearchPaginationQuery($count: Int!, $cursor: String, $query: TeacherSearchQuery!) {
          newSearch {
            teachers(query: $query, first: $count, after: $cursor) {
              didFallback
              resultCount
              pageInfo { hasNextPage endCursor }
              edges {
                node { id legacyId firstName lastName numRatings avgRating }
              }
            }
          }
        }
        """

        # Lowered count to 20 and added fallback: True to prevent API clamping
        variables = {
            "count": 20,
            "cursor": "",
            "query": {
                "text": "",
                "schoolID": self.graphql_school_id,
                "fallback": True
            }
        }

        has_next_page = True
        pages_scraped = 0

        print("Gathering master list of professors from the server...")

        while has_next_page:
            response = requests.post(
                "https://www.ratemyprofessors.com/graphql",
                headers=self.headers, json={"query": query, "variables": variables}, impersonate="chrome"
            )

            try:
                data = response.json()
            except Exception:
                print("Failed to parse JSON. Cloudflare might be blocking the request.")
                break

            if "errors" in data:
                print("GraphQL Error:", data["errors"])
                break

            teachers_data = data.get("data", {}).get("newSearch", {}).get("teachers", {})
            if not teachers_data: break

            total_expected = teachers_data.get("resultCount", "Unknown")

            for edge in teachers_data["edges"]:
                node = edge["node"]
                if node["numRatings"] > 0:
                    prof = Professor(node["legacyId"], node["firstName"], node["lastName"], node["numRatings"], node["avgRating"])
                    professors[prof.ratemyprof_id] = prof

            pages_scraped += 1
            if pages_scraped % 10 == 0:
                print(f"--> Found {len(professors)} professors with ratings so far (Out of ~{total_expected} total)...")

            has_next_page = teachers_data["pageInfo"]["hasNextPage"]
            variables["cursor"] = teachers_data["pageInfo"]["endCursor"]

            # Brief pause so we don't trigger rate limits while paginating the massive list
            time.sleep(0.1)

        print(f"Finished. Successfully compiled a list of {len(professors)} professors with ratings.")
        return professors

    def create_reviews_list(self, tid):
        reviews_list = []
        teacher_string = f"Teacher-{tid}"
        graphql_tid = base64.b64encode(teacher_string.encode('utf-8')).decode('utf-8')

        query = """
        query TeacherRatingsPaginationQuery($count: Int!, $cursor: String, $id: ID!) {
          node(id: $id) {
            ... on Teacher {
              ratings(first: $count, after: $cursor) {
                pageInfo { hasNextPage endCursor }
                edges {
                  node {
                    class
                    comment
                    difficultyRating
                    clarityRating
                    date
                    grade
                    ratingTags
                    wouldTakeAgain
                  }
                }
              }
            }
          }
        }
        """
        variables = {"count": 100, "cursor": "", "id": graphql_tid}

        has_next_page = True
        while has_next_page:
            response = requests.post(
                "https://www.ratemyprofessors.com/graphql",
                headers=self.headers, json={"query": query, "variables": variables}, impersonate="chrome"
            )
            try:
                ratings_data = response.json()["data"]["node"]["ratings"]
            except (TypeError, KeyError):
                break

            for edge in ratings_data["edges"]:
                reviews_list.append(edge["node"])

            has_next_page = ratings_data["pageInfo"]["hasNextPage"]
            variables["cursor"] = ratings_data["pageInfo"]["endCursor"]

        return reviews_list

    def get_all_university_reviews(self) -> pd.DataFrame:
        all_reviews = []
        total_profs = len(self.professorlist)

        print(f"\nFetching text reviews for {total_profs} professors. This will take several minutes...")

        for index, prof in enumerate(self.professorlist):
            if index % 50 == 0 and index > 0:
                print(f"--> Processed reviews for {index}/{total_profs} professors...")

            prof_reviews = self.create_reviews_list(prof.ratemyprof_id)

            for review in prof_reviews:

                # Format the "Would Take Again" response cleanly
                wta_raw = review.get("wouldTakeAgain")
                if wta_raw is True or wta_raw == 1:
                    wta_clean = "Yes"
                elif wta_raw is False or wta_raw == 0:
                    wta_clean = "No"
                else:
                    wta_clean = "N/A"

                # Fix the tags so they format as "Tag 1, Tag 2" instead of parsing every letter
                tags_raw = review.get("ratingTags")
                if isinstance(tags_raw, str):
                    # RMP sends tags separated by "--"
                    tags_clean = tags_raw.replace("--", ", ")
                elif isinstance(tags_raw, list):
                    # Fallback just in case RMP changes their API to send a true list
                    tags_clean = ", ".join(tags_raw)
                else:
                    tags_clean = ""

                review_data = {
                    "professor_id": prof.ratemyprof_id,
                    "professor_name": prof.name,
                    "course": review.get("class"),
                    "review_text": review.get("comment"),
                    "grade": review.get("grade"),
                    "difficulty": review.get("difficultyRating"),
                    "quality": review.get("clarityRating"),
                    "would_take_again": wta_clean,
                    "tags": tags_clean,
                    "date": review.get("date")
                }
                all_reviews.append(review_data)

            time.sleep(0.1)

        return pd.DataFrame(all_reviews)

# --- RUN THE SCRAPE ---
if __name__ == '__main__':
    # 121 is American University's School ID
    au_id = "32"

    print("Initializing scraper for American University...")
    au_api = RateMyProfApi(au_id)

    df_all_reviews = au_api.get_all_university_reviews()

    print("\nScraping complete!")
    print(f"Total reviews extracted: {len(df_all_reviews)}")

    display(df_all_reviews.head(10))

## Cell 2: Load the Dataset
Load the pre-scraped CSV file. If you haven't scraped data yet, run Cell 1 first.

In [ ]:
import pandas as pd

# Load the dataset
# If you don't have au_reviews.csv, run the scraper cell above first to generate it.
# The expected path is data/au_reviews.csv relative to the project root.
df = pd.read_csv('data/au_reviews.csv')
df.head()

In [ ]:
import pandas as pd
import re
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import lime
from lime.lime_text import LimeTextExplainer
import numpy as np

## Cell 3: Preprocessing — Cleaning, Tokenization, and Train/Test Split
- Combines review text + tags into a single `full_text` field
- Cleans text to lowercase letters only (strips punctuation and numbers)
- Builds a vocabulary of the top 10,000 words
- Pads all sequences to 150 tokens
- Splits 80/20 for training/testing

In [ ]:
import pandas as pd
import numpy as np
import re
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split



# Drop rows missing the core review text
df = df.dropna(subset=['review_text'])

# 2. Feature Engineering: Combine Text and Tags
# The tags (e.g., "Tough Grader") are highly predictive, so we append them to the review
df['tags'] = df['tags'].fillna('')
df['full_text'] = df['review_text'] + " " + df['tags']

# 3. Text Cleaning Function
def clean_for_nn(text):
    text = str(text).lower()
    # Regex to keep ONLY lowercase letters and spaces (strips punctuation and numbers)
    text = re.sub(r'[^a-z\s]', '', text)
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Apply the cleaning function
df['nn_cleaned_text'] = df['full_text'].apply(clean_for_nn)

# 4. Tokenization (Building the Vocabulary)
MAX_WORDS = 10000 # We will only keep the 10,000 most frequent words in the dataset
MAX_LEN = 150     # We will standardize all reviews to exactly 150 words

# Initialize tokenizer with an Out-Of-Vocabulary (OOV) token
tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tokenizer.fit_on_texts(df['nn_cleaned_text'])

# Convert the text strings into lists of integers
sequences = tokenizer.texts_to_sequences(df['nn_cleaned_text'])

# 5. Padding
# Force all sequences to be 150 integers long (pads with 0s at the end if too short, truncates if too long)
X_features = pad_sequences(sequences, maxlen=MAX_LEN, padding='post')

# 6. Define Targets
# We are predicting the continuous values (1.0 to 5.0)
y_quality = df['quality'].values
y_difficulty = df['difficulty'].values

# 7. Train/Test Split
# We split the data once, but keep both targets aligned
X_train, X_test, y_q_train, y_q_test, y_d_train, y_d_test = train_test_split(
    X_features, y_quality, y_difficulty, test_size=0.2, random_state=42
)

print(f"Vocabulary Size: {len(tokenizer.word_index)}")
print(f"Shape of X_train (Feature Matrix): {X_train.shape}")
print(f"Shape of y_q_train (Quality Targets): {y_q_train.shape}")
print(f"Shape of y_d_train (Difficulty Targets): {y_d_train.shape}")

Vocabulary Size: 24225
Shape of X_train (Feature Matrix): (27874, 150)
Shape of y_q_train (Quality Targets): (27874,)
Shape of y_d_train (Difficulty Targets): (27874,)


## Cell 4: Multi-Task CNN — Architecture & Training

**Architecture:**
- Embedding(10000 → 100d) → Conv1D(128 filters, k=5) → GlobalMaxPool → Dropout(0.5)
- Two separate Dense(64) → Dense(1, linear) heads for Quality and Difficulty
- 1,080,770 total parameters

**Training:** 10 epochs, Adam optimizer, MSE loss, batch size 64.

> **Note:** The model begins overfitting after epoch ~3 (validation loss rises). Early stopping around epoch 3-4 would improve generalization. The saved weights at epoch 10 overfit the training data.

In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout
import matplotlib.pyplot as plt

# 1. Define the Shared Layers (Learning the Language)
# The input matches your MAX_LEN
input_layer = Input(shape=(MAX_LEN,), name='Text_Input')

# The embedding layer turns your integer IDs into dense, 100-dimensional vectors
# input_dim matches your MAX_WORDS (10000)
shared_embedding = Embedding(input_dim=MAX_WORDS, output_dim=100, input_length=MAX_LEN)(input_layer)

# The Convolutional layer slides a window of 5 words (kernel_size=5) across the text
# This acts as an automated 5-gram feature extractor
shared_conv = Conv1D(filters=128, kernel_size=5, activation='relu')(shared_embedding)

# Global Max Pooling extracts the strongest signal from the convolutions
shared_pool = GlobalMaxPooling1D()(shared_conv)
shared_dropout = Dropout(0.5)(shared_pool) # Dropout prevents overfitting

# 2. Define the Output Branches (Learning the Specific Tasks)
# --- Quality Head ---
q_dense = Dense(64, activation='relu')(shared_dropout)
# Activation is 'linear' because we are predicting a continuous number (1.0 to 5.0)
quality_output = Dense(1, activation='linear', name='Quality_Output')(q_dense)

# --- Difficulty Head ---
d_dense = Dense(64, activation='relu')(shared_dropout)
difficulty_output = Dense(1, activation='linear', name='Difficulty_Output')(d_dense)

# 3. Compile the Model
model = Model(inputs=input_layer, outputs=[quality_output, difficulty_output])

# We use Mean Squared Error (MSE) for the loss function to penalize large errors heavily
# We track Mean Absolute Error (MAE) because it is easier for humans to interpret
# (e.g., an MAE of 0.5 means the model is off by half a point on average)
model.compile(
    optimizer='adam',
    loss={'Quality_Output': 'mse', 'Difficulty_Output': 'mse'},
    metrics={'Quality_Output': 'mae', 'Difficulty_Output': 'mae'}
)

print(model.summary())

# 4. Train the Model
print("\nStarting Training...")
history = model.fit(
    X_train,
    {'Quality_Output': y_q_train, 'Difficulty_Output': y_d_train},
    validation_data=(X_test, {'Quality_Output': y_q_test, 'Difficulty_Output': y_d_test}),
    epochs=10,        # 10 passes through the data
    batch_size=64,    # Processes 64 reviews at a time
    verbose=1
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ Text_Input          │ (None, 150)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 150, 100)  │  1,000,000 │ Text_Input[0][0]  │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 146, 128)  │     64,128 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 128)       │          0 │ conv1d[0][0]      │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 128)       │          0 │ global_max_pooli… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 64)        │      8,256 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 64)        │      8,256 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Quality_Output      │ (None, 1)         │         65 │ dense[0][0]       │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Difficulty_Output   │ (None, 1)         │         65 │ dense_1[0][0]     │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,080,770 (4.12 MB)

 Trainable params: 1,080,770 (4.12 MB)

 Non-trainable params: 0 (0.00 B)

None

Starting Training...
Epoch 1/10
436/436 ━━━━━━━━━━━━━━━━━━━━ 17s 15ms/step - Difficulty_Output_loss: 1.5750 - Difficulty_Output_mae: 0.9994 - Quality_Output_loss: 1.7184 - Quality_Output_mae: 1.0244 - loss: 3.2949 - val_Difficulty_Output_loss: 1.0414 - val_Difficulty_Output_mae: 0.8327 - val_Quality_Output_loss: 0.8788 - val_Quality_Output_mae: 0.7465 - val_loss: 1.9201
Epoch 2/10
436/436 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - Difficulty_Output_loss: 1.0877 - Difficulty_Output_mae: 0.8449 - Quality_Output_loss: 0.8837 - Quality_Output_mae: 0.7441 - loss: 1.9712 - val_Difficulty_Output_loss: 0.9912 - val_Difficulty_Output_mae: 0.8081 - val_Quality_Output_loss: 0.7354 - val_Quality_Output_mae: 0.6570 - val_loss: 1.7263
Epoch 3/10
436/436 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - Difficulty_Output_loss: 0.9658 - Difficulty_Output_mae: 0.7917 - Quality_Output_loss: 0.7255 - Quality_Output_mae: 0.6680 - loss: 1.6909 - val_Difficulty_Output_loss: 0.9472 - val_Difficulty_Output_mae: 0.7840 - val_Q

## Cell 5: Explainability with LIME

LIME (Local Interpretable Model-agnostic Explanations) shows which words drive each prediction.

**The 2D array hack:** LIME expects a classifier with 2+ classes, but our CNN outputs continuous regression scores. We duplicate the continuous prediction into a 2D array `[[score, score], ...]` so LIME's internals work, then explain "label 1" (the duplicated column).

Cells below show Difficulty and Quality explanations for the same review.

### LIME: Difficulty Prediction

In [ ]:
import lime
from lime.lime_text import LimeTextExplainer
import numpy as np

# 1. Initialize Explainer (No 'mode' argument!)
# We provide two "class" names to match our 2D array hack.
explainer = LimeTextExplainer(class_names=['Ignored', 'Difficulty_Score'])

# 2. Build the Prediction Wrapper
def predict_difficulty_for_lime(texts):
    # Apply the same cleaning function
    cleaned_texts = [clean_for_nn(text) for text in texts]

    # Tokenize and pad
    seq = tokenizer.texts_to_sequences(cleaned_texts)
    padded = pad_sequences(seq, maxlen=MAX_LEN, padding='post')

    # Get model predictions (verbose=0 hides the progress bar spam)
    predictions = model.predict(padded, verbose=0)

    # Extract Difficulty predictions and flatten to 1D
    difficulty_preds = predictions[1].reshape(-1)

    # THE HACK: Duplicate our continuous score into a 2D array
    # Column 0: Ignored | Column 1: Difficulty_Score
    return np.vstack((difficulty_preds, difficulty_preds)).T

# 3. Pick a Test Review to Analyze
sample_index = 1   ####CHANGE THIS TO GET DIFFERENT REVIEWS
sample_text = df['full_text'].iloc[sample_index]
actual_difficulty = df['difficulty'].iloc[sample_index]

print(f"Original Text: {sample_text}")
print(f"Actual Student Difficulty Rating: {actual_difficulty}")

# 4. Generate the Explanation!
# We explicitly tell LIME to explain "label 1" (our Difficulty_Score column)
exp = explainer.explain_instance(
    sample_text,
    predict_difficulty_for_lime,
    labels=(1,),
    num_features=6
)

# 5. Display the Results
print("\n--- LIME EXPLANATION ---")
predicted_score = predict_difficulty_for_lime([sample_text])[0][1]
print(f"Model Predicted Difficulty: {predicted_score:.2f}")

print("\nTop words driving this prediction:")
# Make sure to pull the list for label 1
for word, weight in exp.as_list(label=1):
    direction = "INCREASED" if weight > 0 else "DECREASED"
    print(f"The word '{word}' {direction} the predicted difficulty by {abs(weight):.3f} points")

# Visualizing it beautifully inside your Colab notebook
exp.show_in_notebook(text=True)

### LIME: Quality Prediction

In [ ]:
# 1. Initialize a new explainer specifically for Quality
explainer_quality = LimeTextExplainer(class_names=['Ignored', 'Quality_Score'])

# 2. Build the Quality Prediction Wrapper
def predict_quality_for_lime(texts):
    cleaned_texts = [clean_for_nn(text) for text in texts]
    seq = tokenizer.texts_to_sequences(cleaned_texts)
    padded = pad_sequences(seq, maxlen=MAX_LEN, padding='post')

    predictions = model.predict(padded, verbose=0)

    # THE ONLY CHANGE: We grab index 0 (Quality) instead of index 1 (Difficulty)
    quality_preds = predictions[0].reshape(-1)

    return np.vstack((quality_preds, quality_preds)).T

# 3. Generate the Explanation for the SAME review
# (Assuming sample_text and sample_index are still loaded from the previous block)
actual_quality = df['quality'].iloc[sample_index]
print(f"Original Text: {sample_text}")
print(f"Actual Student Quality Rating: {actual_quality}")

exp_quality = explainer_quality.explain_instance(
    sample_text,
    predict_quality_for_lime,
    labels=(1,),
    num_features=6
)

# 4. Display the Results
print("\n--- LIME EXPLANATION (QUALITY) ---")
predicted_q_score = predict_quality_for_lime([sample_text])[0][1]
print(f"Model Predicted Quality: {predicted_q_score:.2f}")

print("\nTop words driving this prediction:")
for word, weight in exp_quality.as_list(label=1):
    direction = "INCREASED" if weight > 0 else "DECREASED"
    print(f"The word '{word}' {direction} the predicted quality by {abs(weight):.3f} points")

# Show the interactive widget
exp_quality.show_in_notebook(text=True)